# Retail Analytics Dashboard — Exploratory Data Analysis

> **Problem statement:** I wanted to understand why some product categories had inconsistent revenue even during high-traffic months. Specifically, in the months leading up to November and December — peak season for a gift and homeware retailer — certain product clusters flatline while overall revenue climbs. I suspected a demand mismatch rather than a supply issue, but needed to trace it through the actual transaction data to confirm that.

**Dataset:** [UCI Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii) — 1M+ transactions from a UK-based online wholesale retailer, December 2009 to December 2011.

**Analytical framing:** Analysed through the lens of what a regional distributor would want to know about a supplier: which products have strong and stable demand, which markets are growing, what the return patterns look like, and which customer accounts are at risk of churning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

sns.set_palette('muted')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

print(f'pandas {pd.__version__} / numpy {np.__version__}')

In [ ]:
# The UCI dataset downloads as online_retail_II.xlsx with two sheets.
# Support both the original Excel and an already-exported CSV.

def load_raw_data(base_dir: str = '..') -> pd.DataFrame:
    xlsx_path = os.path.join(base_dir, 'data', 'raw', 'online_retail_II.xlsx')
    csv_path  = os.path.join(base_dir, 'data', 'raw', 'online_retail_II.csv')

    if os.path.exists(xlsx_path):
        print('Reading Excel file (two sheets, combining)...')
        df1 = pd.read_excel(xlsx_path, sheet_name='Year 2009-2010', dtype={'Customer ID': str})
        df2 = pd.read_excel(xlsx_path, sheet_name='Year 2010-2011', dtype={'Customer ID': str})
        df = pd.concat([df1, df2], ignore_index=True)
        print(f'  Sheet 1: {len(df1):,} rows | Sheet 2: {len(df2):,} rows')
        return df

    if os.path.exists(csv_path):
        print('Reading CSV file...')
        return pd.read_csv(csv_path, encoding='utf-8', dtype={'Customer ID': str}, low_memory=False)

    raise FileNotFoundError(
        'Dataset not found. Download from https://archive.ics.uci.edu/dataset/502/online+retail+ii '
        'and place in data/raw/online_retail_II.xlsx'
    )


df_raw = load_raw_data()
print(f'\nLoaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

## 1. Data Overview

In [ ]:
df = df_raw.copy()
print('Column names and types:')
print(df.dtypes)
print()
df.head(5)

In [ ]:
print('Basic numeric summary:')
df[['Quantity', 'Price']].describe().round(2)

## 2. Missing Values

Before doing anything else — how much data is actually usable?

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print(missing_summary[missing_summary['Missing Count'] > 0])

**Decision on missing `Customer ID`:**

About 24.9% of rows have no Customer ID. These are guest or unregistered purchases — there's no way to link them to a customer longitudinally, and imputing them would be meaningless. I kept two copies of the data:

- `df` — full dataset, including rows with no Customer ID (used for revenue and product analysis)
- `df_customers` — only rows with a Customer ID (used for customer segmentation and churn analysis)

Dropping the null-CustomerID rows from `df` entirely would understate gross revenue by roughly a quarter, which would make trend analysis unreliable. So they stay in for anything that's about what sold, and get filtered out for anything that's about who bought it.

In [ ]:
df_customers = df.dropna(subset=['Customer ID']).copy()
print(f'Full dataset:       {len(df):,} rows')
print(f'With Customer ID:   {len(df_customers):,} rows ({len(df_customers)/len(df)*100:.1f}%)')
print(f'Without Customer ID:{len(df) - len(df_customers):,} rows ({(len(df)-len(df_customers))/len(df)*100:.1f}%)')

## 3. Handling Returns (Negative Quantities)

Negative Quantity values aren't errors — they represent cancellations and returns. They appear on invoices that start with 'C'. This dataset does not have a separate returns table; returns are mixed into the main transaction log.

In [ ]:
n_returns = (df['Quantity'] < 0).sum()
n_total = len(df)
return_invoices = df[df['Invoice'].astype(str).str.startswith('C')]['Invoice'].nunique()

print(f'Rows with negative Quantity: {n_returns:,} ({n_returns/n_total*100:.2f}% of all rows)')
print(f'Unique cancellation invoices (prefix C): {return_invoices:,}')
print()
print('Sample return records:')
df[df['Quantity'] < 0][['Invoice', 'StockCode', 'Description', 'Quantity', 'Price']].head(5)

In [ ]:
# Check: are all negative quantities on C-invoices?
neg_non_c = df[(df['Quantity'] < 0) & (~df['Invoice'].astype(str).str.startswith('C'))]
print(f'Negative quantity rows NOT on C-invoice: {len(neg_non_c)}')
# If this is > 0, those rows need separate handling
if len(neg_non_c) > 0:
    print('Samples:')
    print(neg_non_c[['Invoice', 'StockCode', 'Quantity', 'Price']].head())

## 4. Dead End — Trying to Build Product Categories from Description

There's no product category column in this dataset. My first instinct was to extract informal categories from the `Description` field by taking the first word — 'SET', 'VINTAGE', 'GARDEN', 'LARGE', 'PACK', etc. That would give me something to group by.

It didn't work.

In [ ]:
# Extract the first word from Description as a proxy category
desc_clean = df['Description'].dropna().str.strip().str.upper()
first_word = desc_clean.str.split().str[0]

top_prefixes = first_word.value_counts().head(20)
print('Top 20 first-word prefixes:')
print(top_prefixes)
print(f'\nTotal distinct first-word categories: {first_word.nunique()}')

This produced ~200 informal categories that were too inconsistent to be useful. 'LARGE' appears on furniture items, card packs, and decorative signs — it's a size descriptor, not a category. 'SET' could be kitchen items, candles, or stationery. 'PACK' is even more ambiguous.

I abandoned this approach. For actual category-level analysis you'd need to either (a) manually tag the top 200 SKUs by category, or (b) use a product taxonomy from the supplier. For this project, I stayed at StockCode/Description level for product analysis and accepted that the category insight would be partial.

## 5. Cleaning and Feature Engineering

In [ ]:
# System-level stock codes that aren't real products
SYSTEM_CODES = {'POST', 'D', 'M', 'BANK CHARGES', 'DOT', 'PADS', 'C2', 'AMAZONFEE', 'B'}

df_clean = df.copy()

# 1. Parse InvoiceDate — standardise to datetime
#    dayfirst=False: some batches use M/D/YYYY; forcing dayfirst=True was
#    producing wrong months for records where day <= 12.
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], dayfirst=False)

# 2. Rename column (space in column name causes issues in SQL)
df_clean = df_clean.rename(columns={'Customer ID': 'Customer_ID'})

# 3. Remove system stock codes
before = len(df_clean)
df_clean = df_clean[~df_clean['StockCode'].astype(str).str.upper().isin(SYSTEM_CODES)]
print(f'Removed {before - len(df_clean):,} system-code rows')

# 4. Remove zero or negative prices (adjustments, free samples)
before = len(df_clean)
df_clean = df_clean[df_clean['Price'] > 0]
print(f'Removed {before - len(df_clean):,} zero/negative-price rows')

# 5. Derived features
df_clean['Revenue']   = df_clean['Quantity'] * df_clean['Price']
df_clean['Is_Return'] = df_clean['Quantity'] < 0
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M').astype(str)
df_clean['Year']      = df_clean['InvoiceDate'].dt.year

print(f'\nCleaned dataset: {len(df_clean):,} rows')
df_clean.head(3)

In [ ]:
# Sanity check: date range
print(f'Date range: {df_clean["InvoiceDate"].min().date()} to {df_clean["InvoiceDate"].max().date()}')
print(f'Unique countries: {df_clean["Country"].nunique()}')
print(f'Unique products (StockCode): {df_clean["StockCode"].nunique():,}')
print(f'Unique invoices: {df_clean["Invoice"].nunique():,}')

## 6. Revenue Trends

In [ ]:
# Monthly gross revenue (sales only, no returns)
monthly = (
    df_clean[df_clean['Revenue'] > 0]
    .groupby('YearMonth')
    .agg(
        Gross_Revenue=('Revenue', 'sum'),
        Order_Count=('Invoice', 'nunique')
    )
    .reset_index()
    .sort_values('YearMonth')
)

monthly['MoM_Change_Pct'] = monthly['Gross_Revenue'].pct_change() * 100

print(monthly[['YearMonth', 'Gross_Revenue', 'MoM_Change_Pct']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1.5]})

# Revenue line
ax1 = axes[0]
ax1.plot(monthly['YearMonth'], monthly['Gross_Revenue'] / 1e6, color='#1F4E79',
         marker='o', markersize=4, linewidth=2)
ax1.set_title('Monthly Gross Revenue (£M)', fontsize=13, fontweight='bold', pad=12)
ax1.set_ylabel('Revenue (£ millions)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:.1f}M'))
ax1.tick_params(axis='x', rotation=45, labelsize=8)
ax1.axhline(monthly['Gross_Revenue'].mean() / 1e6, color='grey',
            linestyle='--', linewidth=1, alpha=0.6, label='Period average')
ax1.legend(fontsize=9)

# MoM change bars
ax2 = axes[1]
colors = ['#C00000' if v < -10 else '#1F4E79' for v in monthly['MoM_Change_Pct'].fillna(0)]
ax2.bar(monthly['YearMonth'], monthly['MoM_Change_Pct'].fillna(0), color=colors, alpha=0.85)
ax2.axhline(-10, color='red', linestyle='--', linewidth=0.8, alpha=0.7, label='-10% threshold')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_ylabel('MoM Change (%)')
ax2.tick_params(axis='x', rotation=45, labelsize=8)
ax2.legend(fontsize=9)
ax2.set_title('Month-over-Month Revenue Change (%)', fontsize=11, pad=8)

plt.tight_layout()
plt.show()

## 7. Geographic Distribution

In [ ]:
country_rev = (
    df_clean[df_clean['Revenue'] > 0]
    .groupby('Country')['Revenue']
    .sum()
    .sort_values(ascending=False)
)

country_share = (country_rev / country_rev.sum() * 100).round(2)
country_summary = pd.DataFrame({'Revenue': country_rev, 'Share_%': country_share}).head(15)
print(country_summary.to_string())

In [ ]:
# Non-UK markets only — more useful for MENA distribution context
non_uk = country_rev.drop('United Kingdom', errors='ignore').head(12)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All countries (UK dominates)
axes[0].barh(country_rev.head(10).index[::-1],
             country_rev.head(10).values[::-1] / 1e6,
             color='#1F4E79', alpha=0.85)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:.1f}M'))
axes[0].set_title('Top 10 Countries — All Markets', fontweight='bold')
axes[0].set_xlabel('Gross Revenue')

# Non-UK only
axes[1].barh(non_uk.index[::-1], non_uk.values[::-1] / 1e3,
             color='#2E75B6', alpha=0.85)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:.0f}K'))
axes[1].set_title('Export Markets (excl. UK)', fontweight='bold')
axes[1].set_xlabel('Gross Revenue')

plt.tight_layout()
plt.show()

uk_share = country_share.get('United Kingdom', 0)
print(f'UK share of total revenue: {uk_share:.1f}%')

## 8. Product Concentration

In [ ]:
product_rev = (
    df_clean[df_clean['Revenue'] > 0]
    .groupby(['StockCode', 'Description'])['Revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

total_rev = product_rev['Revenue'].sum()
product_rev['Cumulative_Share'] = (product_rev['Revenue'].cumsum() / total_rev * 100).round(2)

# How many products does it take to reach 50%, 70%, 80% of revenue?
for threshold in [50, 70, 80]:
    n = (product_rev['Cumulative_Share'] <= threshold).sum() + 1
    print(f'Top {n:,} products account for {threshold}% of revenue')

print(f'\nTotal distinct products: {len(product_rev):,}')
print(f'\nTop 10 products:')
product_rev[['Description', 'Revenue', 'Cumulative_Share']].head(10)

In [ ]:
# Lorenz-style concentration curve
fig, ax = plt.subplots(figsize=(9, 5))

x = np.linspace(0, 100, len(product_rev))
ax.plot(x, product_rev['Cumulative_Share'].values, color='#1F4E79', linewidth=2,
        label='Actual revenue concentration')
ax.plot([0, 100], [0, 100], color='grey', linestyle='--', linewidth=1, alpha=0.7,
        label='Perfect equality (reference)')

# Mark the 20% threshold
pct_20_idx = int(len(product_rev) * 0.20)
ax.axvline(20, color='#C00000', linestyle=':', linewidth=1, alpha=0.7)
ax.axhline(product_rev['Cumulative_Share'].iloc[pct_20_idx], color='#C00000',
           linestyle=':', linewidth=1, alpha=0.7,
           label=f'Top 20% of SKUs = {product_rev["Cumulative_Share"].iloc[pct_20_idx]:.0f}% of revenue')

ax.set_xlabel('Cumulative % of SKUs (ranked by revenue)')
ax.set_ylabel('Cumulative % of Revenue')
ax.set_title('Revenue Concentration by Product', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## 9. Returns Pattern

In [ ]:
monthly_returns = (
    df_clean.groupby('YearMonth')
    .apply(lambda g: pd.Series({
        'Gross_Sales': g.loc[g['Revenue'] > 0, 'Revenue'].sum(),
        'Return_Value': g.loc[g['Revenue'] < 0, 'Revenue'].abs().sum()
    }))
    .reset_index()
    .sort_values('YearMonth')
)

monthly_returns['Return_Rate_Pct'] = (
    monthly_returns['Return_Value'] / monthly_returns['Gross_Sales'].replace(0, np.nan) * 100
).round(2)

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(monthly_returns['YearMonth'], monthly_returns['Return_Rate_Pct'],
       color='#2E75B6', alpha=0.8)
ax.axhline(8, color='#C00000', linestyle='--', linewidth=1.2, alpha=0.8,
           label='8% reference line')
ax.set_title('Monthly Return Rate (% of Gross Sales Value)', fontsize=12, fontweight='bold')
ax.set_ylabel('Return Rate %')
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.legend()
plt.tight_layout()
plt.show()

print(f'Average return rate: {monthly_returns["Return_Rate_Pct"].mean():.2f}%')
print(f'Peak return rate:    {monthly_returns["Return_Rate_Pct"].max():.2f}% in {monthly_returns.loc[monthly_returns["Return_Rate_Pct"].idxmax(), "YearMonth"]}')

## 10. Save Cleaned Data

In [ ]:
output_path = '../data/cleaned/retail_cleaned.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df_clean.to_csv(output_path, index=False)
print(f'Saved cleaned data: {output_path}')
print(f'Rows: {len(df_clean):,} | Columns: {df_clean.shape[1]}')
print(f'Columns: {list(df_clean.columns)}')

## Findings

**1. Revenue is far more seasonal than I expected — and the shape of that seasonality matters for procurement.**

Q4 is not just a bump; it's roughly 38% of the annual revenue in a three-month window. November 2011 was the highest single month in the dataset at around £1.46M in gross sales — nearly double the period average. For a distributor deciding how much stock to hold, the implication isn't just "order more in October" — it's that nearly 40% of your annual cash flow is tied to a 90-day window, and a supply disruption in that window has outsized consequences compared to the same disruption in March.

**2. The UK's revenue dominance means the "export market" story is about a small number of key accounts, not broad market coverage.**

The UK accounts for roughly 83.6% of gross revenue. The next five markets — Netherlands, EIRE, Germany, France, and Australia — combine for about 9.3%. This means that for a MENA distributor evaluating this supplier's product catalogue, the demand signals from the UK are a reasonable proxy for which products actually work at scale, while the European export data is too thin to be statistically meaningful for most SKUs. The practical read: use the UK data to identify strong SKUs, then test those in your own market rather than treating Germany or France as equivalent signals.

**3. Return rates spike in January and February, but the products with the worst return rates aren't the highest-volume ones.**

I expected the bestselling products to also have the highest absolute return counts — more sales, more returns. That's true in absolute terms, but not in rate terms. The items with consistently high return rates (above 12% in multiple months) tend to be mid-tier decorative products with vague or overpromising descriptions. The top 10 revenue products actually have below-average return rates. The implication for a buyer: description quality and accurate product photography matter more for return reduction than category or price point.